# ЛР 05 — Практика 1: Drift detection и monitoring quality

## Перед началом
- **Для кого эта ЛР**: для студента, который только начинает работать с мониторингом моделей.
- **Зачем это в реальной жизни**: модель может внешне выглядеть стабильной, но качество уже падает на новых данных.
- **Что получится в конце**: вы получите два контрактных артефакта — `drift_detection_audit.csv` и `monitoring_quality_audit.csv`.
- **Что можно не знать заранее**: строгие доказательства статистических тестов и продвинутые методы MLOps.
- **Что делать, если застрял**: вернитесь к теории (разделы 0-5), проверьте блоки "Вход/Выход" и ответьте на "Проверь себя" текущего шага.

## Как работать с этим ноутбуком
- Идем строго по шагам: вход -> расчет -> интерпретация -> экспорт.
- После каждого шага фиксируем мини-вывод простыми словами.

## Мост из теории к практике
- В теории вы узнали смысл терминов, а здесь увидите их в реальных таблицах.
- Цепочка практики 1: **сигнал в данных -> объяснение сигнала -> подготовка входа для policy**.


## Шаг 1. Подготовка окружения и входов
- **Что уже знаем**: корректное сравнение возможно только относительно `reference` и фиксированного формата данных.
- **Что новое в этом шаге**: загружаем оба датасета курса и проверяем базовый контракт.
- **Зачем это в проекте**: если вход поврежден, все дальнейшие выводы будут недостоверны.
- **Теоретическая опора (где это применится через 5 минут)**: разделы 0-2 теории.
- **Вход**: исходные CSV `medical`, `finance`.
- **Выход**: словарь `datasets`.
- **Переход к следующему шагу**: из `datasets` построим две аудит-таблицы мониторинга.

### Термины шага (на пальцах)
- **Эталонный срез** (`reference`): база, с которой сравниваем новые окна.
- **Воспроизводимость** (`random_state`): одинаковый запуск дает одинаковый результат.
- **Сохранение долей классов** (`stratify`): защита от перекоса train/test при разбиении.
- **Типичная ошибка новичка**: считать, что "и так сойдет", и не проверять наличие `target`.
- **Короткий контрпример**: в одном файле нет `target`, но код продолжили — все метрики дальше невалидны.

### Проверь себя
- **Ожидаемый формат ответа**: (1) перечислите загруженные датасеты; (2) подтвердите наличие колонки `target`.

### Мини-вывод
- Входные данные прошли базовую проверку, можно переходить к расчету сигналов.

### Как интерпретировать результат новичку
- **Что вижу в таблице**: в `datasets` есть оба датасета с корректной структурой.
- **Что это значит**: стартовый контракт соблюден.
- **Что делаю дальше**: запускаю расчет drift и quality без ручных "костылей".

### TODO(обязательно)
- Объясните в 2-3 предложениях, почему ЛР05 автономна и не требует артефактов ЛР03/ЛР04.
1. **Разные задачи**: ЛР03-04 занимаются отбором признаков и обучением моделей, а ЛР05 — мониторингом уже работающих моделей во времени.
2. **Независимые входы**: ЛР05 использует только исходные данные и утилиты из `lab_utils.py`, пересчитывая все метрики "с нуля" для каждого окна мониторинга.
3. **Проверка на реальных данных**: Для мониторинга нужно видеть эволюцию распределений во времени, а не фиксированный набор отобранных признаков.


In [1]:
# Что делаем:
# - Подключаем `lab_utils` и загружаем оба датасета курса.
# Зачем:
# - Формируем единый и воспроизводимый вход для всего дальнейшего workflow.
# Как читать результат:
# - Новичку важно увидеть оба ключа (`medical`, `finance`) и убедиться, что данные реально загрузились.
# Типичные ошибки:
# - Ошибка пути к `lab_utils.py`; исправление: используйте fallback на `../lab_utils.py`.
from pathlib import Path

import pandas as pd

import importlib.util

module_path = Path("lab_utils.py")
if not module_path.exists():
    module_path = Path("../lab_utils.py")

spec = importlib.util.spec_from_file_location("lab05_utils", module_path)
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)
MODULE_ROOT = module_path.parent.resolve()

datasets = lab.load_course_datasets()
sorted(datasets.keys())


['finance', 'medical']

## Шаг 2. Расчет drift_detection_audit и monitoring_quality_audit
- **Что уже знаем**: drift нельзя читать в отрыве от качества и стоимости ошибок.
- **Что новое в этом шаге**: строим оба обязательных артефакта по всем сценариям.
- **Зачем это в проекте**: именно эти таблицы станут фактической основой policy-решения.
- **Теоретическая опора (где это применится через 5 минут)**: разделы 3-5 теории.
- **Вход**: `datasets`, `lab.build_full_monitoring_cycle`.
- **Выход**: `drift_detection_audit`, `monitoring_quality_audit`.
- **Переход к следующему шагу**: агрегируем сигналы в компактную сводку риска.

### Термины шага (на пальцах)
- **Статистическая заметность** (`p-value`): насколько отличие похоже на реальный сигнал, а не шум.
- **Сила сдвига** (`PSI`): насколько изменение велико на практике.
- **Флаг сдвига** (`drift_flag`): объединенный индикатор по признаку.
- **Типичная ошибка новичка**: смотреть только на один показатель (`p-value` или `PSI`).
- **Короткий контрпример**: `p-value` маленький, но `PSI` и `delta_cost_vs_reference` маленькие — немедленный `retrain` не обязателен.

### Теоретическая карточка (перед выполнением)
- Чтение сигнала всегда двойное: `p_value` + `effect_size` (`PSI`).
- Затем проверяем, отражается ли это на `f1` и `expected_cost`.
- Только после этого говорим о риске сценария.

### Проверь себя
- **Ожидаемый формат ответа**: (1) перечислите сценарии из `scenario`; (2) подтвердите, что обе таблицы непустые.

### Мини-вывод
- Получены две контрактные таблицы, в которых уже видно, где сдвиг заметнее.

### Как интерпретировать результат новичку
- **Что вижу в таблице**: появились строки по `stable/covariate/prior/combined` и всем признакам/моделям.
- **Что это значит**: мониторинговый расчет завершен корректно.
- **Что делаю дальше**: перехожу к сводке, чтобы выделить приоритетные риски.

### TODO(обязательно)
- Укажите 1-2 сценария, которые кажутся наиболее рискованными, и назовите подтверждающие колонки.
1. combined для обоих датасетов — потому что здесь одновременно меняются и признаки (covariate), и целевая переменная (prior), что дает наибольшую неопределенность.
2. prior для finance — сдвиг доли дефолтных заемщиков напрямую влияет на стоимость ошибок.

Подтверждающие колонки: drift_feature_share (доля сдвинутых признаков) и delta_cost_vs_reference (рост стоимости ошибок).


In [2]:
# Что делаем:
# - Строим две аудит-таблицы по каждому датасету и объединяем результаты.
# Зачем:
# - Без этих таблиц нельзя перейти к доказуемому policy-решению.
# Как читать результат:
# - Новичку важно проверить наличие строк по всем сценариям и обоим датасетам.
# Типичные ошибки:
# - Потерять часть строк при объединении; исправление: используйте `ignore_index=True`.
drift_frames = []
quality_frames = []

for dataset_name, frame in datasets.items():
    drift, quality, _, _ = lab.build_full_monitoring_cycle(
        dataset_name=dataset_name,
        df=frame,
        window_size=220,
        random_state=lab.SEED,
    )
    drift_frames.append(drift)
    quality_frames.append(quality)

drift_detection_audit = pd.concat(drift_frames, ignore_index=True)
monitoring_quality_audit = pd.concat(quality_frames, ignore_index=True)

drift_detection_audit.head()


,dataset,window_id,scenario,feature,feature_type,detector,statistic,p_value,effect_size,drift_flag
0,medical,1,stable,age,numeric,ks,0.012698,1.000000,0.004450,False
1,medical,1,stable,sex,categorical,chi2,0.000000,1.000000,0.000024,False
2,medical,1,stable,bmi,numeric,ks,0.045742,0.942095,0.031500,False
3,medical,1,stable,systolic_bp,numeric,ks,0.024242,0.999995,0.010017,False
4,medical,1,stable,diastolic_bp,numeric,ks,0.028716,0.999784,0.012154,False


## Шаг 3. Быстрая интерпретация мониторинга
- **Что уже знаем**: у нас есть детальные сигналы по каждому признаку и сценарию.
- **Что новое в этом шаге**: превращаем детальные строки в управленческую сводку риска.
- **Зачем это в проекте**: policy в практике 2 опирается на агрегированные признаки риска.
- **Теоретическая опора (где это применится через 5 минут)**: разделы 5, 6 и 9 теории.
- **Вход**: `drift_detection_audit`, `monitoring_quality_audit`.
- **Выход**: сводка `drift_feature_share` + `delta_cost_vs_reference`.
- **Переход к следующему шагу**: экспортируем контрактные CSV для практики 2.

### Термины шага (на пальцах)
- **Доля сдвинутых признаков** (`drift_feature_share`): какая часть признаков помечена как проблемная.
- **Рост стоимости** (`delta_cost_vs_reference`): насколько дороже стали ошибки относительно `reference`.
- **Типичная ошибка новичка**: смотреть только среднее и игнорировать худший сценарий.
- **Короткий контрпример**: средняя деградация мягкая, но один сценарий резко повышает стоимость.

### Теоретическая карточка (перед выполнением)
- Этот шаг — мост от статистики к управленческому решению.
- Сначала ранжируем сценарии по `drift_feature_share`.
- Затем сверяем, где растет `delta_cost_vs_reference`.

### Проверь себя
- **Ожидаемый формат ответа**: (1) назовите сценарий с максимальным `drift_feature_share`; (2) сценарий с максимальным `delta_cost_vs_reference`.

### Мини-вывод
- Сценарные риски ранжированы, понятно, куда смотреть в policy в первую очередь.

### Как интерпретировать результат новичку
- **Что вижу в таблице**: по каждому сценарию есть числовой риск по drift и стоимости.
- **Что это значит**: можно обосновать приоритеты без "ощущений".
- **Что делаю дальше**: сохраняю результаты как вход для практики 2.

### TODO(обязательно)
- Опишите связь между `drift_feature_share` и `delta_cost_vs_reference` на ваших данных.

Связь между drift_feature_share и delta_cost_vs_reference:
- Ожидаемая связь: чем больше признаков "уехало" (высокий drift_feature_share), тем выше риск роста стоимости ошибок.

- На практике: иногда даже при высоком drift_feature_share стоимость может оставаться стабильной, если сдвинулись неинформативные признаки или модель устойчива к изменениям.

- Вывод: оба показателя нужно смотреть вместе — drift_feature_share дает раннее предупреждение, а delta_cost_vs_reference подтверждает бизнес-последствия.


In [3]:
# Что делаем:
# - Агрегируем долю drift-флагов и worst-case рост стоимости по сценариям.
# Зачем:
# - Это прямой мост от статистического сигнала к управленческому риску.
# Как читать результат:
# - Новичку: высокий `drift_feature_share` вместе с высоким `delta_cost` = сценарий повышенного внимания.
# Типичные ошибки:
# - Усреднять `delta_cost` и терять худший случай; исправление: используйте `max()`.
drift_summary = (
    drift_detection_audit.groupby(["dataset", "scenario"], as_index=False)["drift_flag"]
    .mean()
    .rename(columns={"drift_flag": "drift_feature_share"})
)

cost_summary = (
    monitoring_quality_audit.groupby(["dataset", "scenario"], as_index=False)["delta_cost_vs_reference"]
    .max()
)

drift_summary.merge(cost_summary, on=["dataset", "scenario"], how="inner")


,dataset,scenario,drift_feature_share,delta_cost_vs_reference
0,finance,combined,0.285714,1.472727
1,finance,covariate,0.214286,0.295455
2,finance,prior,0.071429,1.200000
3,finance,stable,0.000000,0.077273
4,medical,combined,0.384615,2.372222
5,medical,covariate,0.230769,0.026768
6,medical,prior,0.230769,2.317677
7,medical,stable,0.000000,0.135859


## Шаг 4. Экспорт артефактов практики 1
- **Что уже знаем**: риск по сценариям оценен и интерпретирован.
- **Что новое в этом шаге**: фиксируем результаты в контрактных CSV.
- **Зачем это в проекте**: без корректного экспорта нельзя воспроизводимо перейти к policy-решению.
- **Теоретическая опора (где это применится через 5 минут)**: разделы 1 и 6 теории.
- **Вход**: `drift_detection_audit`, `monitoring_quality_audit`.
- **Выход**: `outputs/drift_detection_audit.csv`, `outputs/monitoring_quality_audit.csv`.
- **Переход к следующему шагу**: открывайте практику 2 и загружайте эти файлы как обязательный вход.

### Термины шага (на пальцах)
- **Контрактный артефакт**: файл с фиксированным именем и набором колонок.
- **Проверка контракта**: структура должна точно совпадать с контрактом ЛР05.
- **Типичная ошибка новичка**: сохранить файл под похожим, но не точным именем.
- **Короткий контрпример**: `monitoring_quality.csv` вместо `monitoring_quality_audit.csv` ломает весь следующий шаг.

### Проверь себя
- **Ожидаемый формат ответа**: (1) укажите путь к двум файлам; (2) перечислите по 2-3 ключевые колонки каждого файла.

### Мини-вывод
- Практика 1 завершена: вход для policy в практике 2 готов.

### Как интерпретировать результат новичку
- **Что вижу в таблице**: оба контрактных CSV сохранены в `outputs/`.
- **Что это значит**: результат можно проверить и воспроизвести.
- **Что делаю дальше**: перехожу в практику 2 для выбора `observe`/`retrain`.

### TODO(обязательно)
- Реализуйте экспорт и удалите intentional-stop.


In [4]:
# Что делаем:
# - Сохраняем две контрактные аудит-таблицы в `outputs/`.
# Зачем:
# - Эти файлы — обязательный вход для практики 2 и последующей проверки.
# Как читать результат:
# - Новичку: после выполнения появятся два CSV с точными именами.
# Типичные ошибки:
# - Сохранение в неверную папку; исправление: используйте `MODULE_ROOT / "outputs"`.
# TODO(обязательно):
# 1) сохраните drift_detection_audit.csv
# 2) сохраните monitoring_quality_audit.csv
# 3) удалите intentional-stop
output_dir = MODULE_ROOT / "outputs"
output_dir.mkdir(exist_ok=True)

drift_detection_audit.to_csv(output_dir / "drift_detection_audit.csv", index=False)
monitoring_quality_audit.to_csv(output_dir / "monitoring_quality_audit.csv", index=False)

print(f"Saved: {output_dir / 'drift_detection_audit.csv'}")
print(f"Saved: {output_dir / 'monitoring_quality_audit.csv'}")


Saved: C:\Users\perev\Documents\GitHub\edu-big-data-machine-models\05-drift-monitoring-and-retraining-policy\outputs\drift_detection_audit.csv
Saved: C:\Users\perev\Documents\GitHub\edu-big-data-machine-models\05-drift-monitoring-and-retraining-policy\outputs\monitoring_quality_audit.csv
